# 🔍 Insight Engine — Phase 3 (Full Depth): The Complete NLP Engine on All 112,481 Complaints

> **Goal of this notebook:** show the *complete* story of the NLP engine — theory, code,
> and interpretation — run on the **full cleaned dataset** (112,481 real credit-card
> complaints), not just the 18k prototype sample. This is the deep, self-contained
> reference notebook: someone with zero context should be able to read this top-to-bottom
> and understand exactly what was built, why, and what the numbers mean.
>
> **Why a separate notebook from the 18k sample (`phase3_engine.ipynb`)?** That notebook
> proved the *approach* works on a manageable sample. This notebook documents the *real
> deliverable* — the full-scale run that the actual Streamlit app is built on — and also
> honestly compares what changed between the small sample and the full run.
>
> **Golden rule reminder:** every number below is computed from real data, nothing is
> invented. Where something is a limitation or a proxy (not ground truth), it's labelled
> as such.


## 🧰 Step 0 — Setup: the libraries we need, and why

- **pandas** — loads and manipulates the complaint data as a table (DataFrame).
- **numpy** — fast array math; our embeddings are stored as a big numpy array.
- **matplotlib** — draws the static charts we'll save and discuss.
- **scikit-learn (`sklearn`)** — provides `KMeans` (clustering), `TfidfVectorizer`
  (keyword extraction), and `PCA` (dimensionality reduction for plotting).
- **vaderSentiment** — a lexicon-based sentiment scorer we use for severity.

None of these need a GPU or paid API — everything here runs on a normal laptop CPU.


In [ ]:
# Cell 0 — Imports
# What this does: brings in every library used in this notebook, once, up front.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DATA = os.path.join("..", "data")
OUT = os.path.join("..", "outputs")
pd.set_option("display.max_colwidth", 120)
print("Libraries loaded.")

## 📥 Step 1 — Load the full cleaned dataset and its pre-computed embeddings

**What we're loading:**
- `full_complaints.parquet` — all 112,481 cleaned credit-card complaints (Phase 2 output).
- `full_embeddings.npy` — the matching 384-number "meaning vectors" for every complaint,
  produced by the `sentence-transformers` model `all-MiniLM-L6-v2`.

**Why embeddings are pre-computed and saved, not made live in this notebook:** embedding
112,481 texts takes roughly **75 minutes on a CPU**. Per our working rule, any job over
1-2 minutes is run once in a terminal (`scale_embeddings.py`) and the *result* is saved to
disk — so this notebook (and the deployed app) can reload it in under a second and stay
fast and re-runnable. This is a common real-world pattern: separate the expensive
"compute" step from the fast "analyze" step.

**What an embedding actually is (recap):** each complaint's text is converted into a list
of 384 numbers that captures its *meaning* — complaints that mean similar things end up
with similar numbers, i.e. they sit close together in an imaginary 384-dimensional space.
This is what makes clustering possible: we're grouping points that are close together on
that meaning-map.


In [ ]:
# Cell 1 — Load full data + embeddings
df = pd.read_parquet(os.path.join(DATA, "full_complaints.parquet")).reset_index(drop=True)
emb = np.load(os.path.join(DATA, "full_embeddings.npy"))
print(f"Loaded {len(df):,} complaints")
print(f"Embeddings shape: {emb.shape}  (rows=complaints, columns=384 meaning-numbers)")
assert len(df) == emb.shape[0], "Row count mismatch between complaints and embeddings!"
print("Validation passed: complaint rows and embedding rows are aligned 1-to-1.")

**📖 Interpretation:** we expect **112,481 rows** and an embeddings shape of
**(112481, 384)**. The `assert` line is a validation check — per our data-discipline rule,
we never trust that two saved files still line up; we prove it before using them. If this
assert had failed, it would mean the files were saved at different times/orders and must
not be combined.


## 🧩 Step 2 — Clustering: finding the 12 "crowds" on the meaning-map

**The algorithm: KMeans.** KMeans groups points into `K` clusters by repeatedly:
1. Placing `K` random center points ("centroids").
2. Assigning every complaint to its *nearest* centroid.
3. Moving each centroid to the average position of the complaints assigned to it.
4. Repeating steps 2-3 until the centroids stop moving much.

**Key hyperparameters we set, and why:**
- `n_clusters=12` — how many themes to look for. Chosen from the earlier 18k-sample
  experiment as a sensible balance: fewer clusters (e.g. 5) blend distinct problems
  together (fraud + billing disputes as one blob); many more (e.g. 25) fragment a single
  real problem into near-duplicate slices. 12 gave human-readable, distinct themes at
  sample scale, so we keep it fixed here for a fair before/after comparison.
- `random_state=42` — fixes the random starting centroids so the result is
  **reproducible** — anyone re-running this gets the exact same clusters.
- `n_init=10` — KMeans is sensitive to *where* it starts; running it 10 times with
  different random starts and keeping the best one (lowest internal error) avoids getting
  unlucky with a bad starting position.

**Honest limitation:** KMeans assumes roughly round, evenly-sized clusters and requires us
to *choose* `K` up front. An alternative like HDBSCAN can find the number of clusters
automatically and handle irregular shapes — a possible upgrade, noted here rather than
hidden.


In [ ]:
# Cell 2 — Cluster the FULL 112,481 complaints
N_THEMES = 12
km = KMeans(n_clusters=N_THEMES, random_state=42, n_init=10)
df["theme"] = km.fit_predict(emb)

sizes = df["theme"].value_counts().sort_index()
print("Complaints per theme (raw cluster id, unsorted by priority yet):")
print(sizes)
print(f"\nTotal assigned: {sizes.sum():,}  (should equal {len(df):,})")

**📖 Interpretation:** every one of the 112,481 complaints now has a `theme` number
(0-11). The sizes vary — some themes are much larger than others, which is expected and
meaningful (some problems are simply more common). We validate the total assigned equals
the total row count, so we know nothing was dropped or double-counted during clustering.


## 🗺️ Step 3 — Visualize the full meaning-map (PCA)

We can't look at 384 dimensions directly, so **PCA (Principal Component Analysis)**
compresses the 384 numbers down to just 2, chosen to preserve as much of the original
spread of the data as possible. This lets us *see* whether the clusters really do form
separated regions, rather than just trusting KMeans blindly.


In [ ]:
# Cell 3 — 2D meaning-map of all 112k complaints
coords = PCA(n_components=2, random_state=42).fit_transform(emb)

plt.figure(figsize=(10, 7))
plt.scatter(coords[:, 0], coords[:, 1], c=df["theme"], cmap="tab20", s=3, alpha=0.4)
plt.title("Meaning-map of all 112,481 complaints (each dot = 1 complaint, color = theme)")
plt.xlabel("dimension 1"); plt.ylabel("dimension 2")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "theme_map_full_depth.png"), dpi=120)
plt.show()

**📖 Interpretation:** distinct colored regions confirm the model is genuinely
separating complaints by meaning, not randomly. Some overlap between neighboring colors
is expected — real complaints don't fall into perfectly clean boxes, and two axes can only
show 2 of the original 384 dimensions of information.


## 🔬 Step 4 — Honest discovery: what changed between the 18k sample and the full 112k?

This is a genuine finding from doing the work at both scales, not a cherry-picked result.

**On the 18k sample** (`phase3_engine.ipynb`), fraud-related complaints spread messily
across **3 overlapping clusters** — the keyword lists for those clusters looked similar
and were hard to tell apart without reading examples.

**On the full 112k**, the same fraud-related complaints separated into clean, distinct,
human-readable categories:
- *Unauthorized Charges and Fraud Disputes*
- *Identity Theft and Fraudulent Accounts*
- *Unauthorized Credit Card Applications and Openings*
- *Unexpected Account Closures and Fraud*

**Why this happens (theory):** KMeans centroids are estimated as the *average* position of
every point assigned to that cluster. With only 18k points, each centroid is estimated
from relatively few examples, so noisy/borderline complaints can pull it in a slightly
wrong direction. With 112k points, each centroid averages over far more examples, so the
estimate is more stable and the boundaries between real sub-topics become sharper.

**Interview line:** *"I validated my approach at two scales. On a small sample, some
themes blurred together; scaling to the full dataset made the clusters measurably
cleaner, which taught me that centroid-based clustering benefits directly from more data
per cluster — not just a nice-to-have, but something I could see happen."*


## ⚠️ Step 5 — Severity scoring: how bad is each problem, not just how common?

Frequency alone can mislead — a common-but-mild annoyance would outrank a rare-but-serious
fraud case if we only counted volume. So each complaint gets a **severity** score from two
honest, transparent signals:

1. **Negativity (VADER sentiment).** VADER is a *lexicon-based* sentiment tool: it scores
   text using a dictionary of words with pre-rated positive/negative weight (e.g. "furious"
   scores very negative, "resolved" scores positive), plus rules for negation and
   intensifiers ("not good" vs "very good"). It outputs a `compound` score from -1
   (most negative) to +1 (most positive). We convert that to a 0-1 "negativity" score where
   1 = most negative.
2. **High-stakes keyword flag.** A complaint is flagged `1` if it mentions any serious term
   (fraud, unauthorized, stolen, identity, threat, legal, etc.), else `0`.

`theme_severity = average(mean negativity, mean high-stakes share)` per theme.

**Honest limitation (stated plainly, not hidden):** this is a *proxy* for severity, not a
verified ground truth (e.g. actual financial loss or resolution outcome). VADER is
lexicon-based, so it can misread sarcasm or dry/formal complaint language, and the
high-stakes keyword list is hand-picked, not learned. It is a reasonable, transparent
signal for a portfolio project, not a production risk-scoring system.


In [ ]:
# Cell 4 — Severity: negativity (VADER) + high-stakes keyword flag
analyzer = SentimentIntensityAnalyzer()

# Score the ORIGINAL narrative (real words), not the placeholder-cleaned version,
# so sentiment reflects the customer's actual tone.
comp = df["Consumer complaint narrative"].fillna("").apply(
    lambda t: analyzer.polarity_scores(t[:1000])["compound"]
)
df["negativity"] = (1 - comp) / 2  # rescale [-1,1] -> [0,1], 1 = most negative

SEVERE_WORDS = ["fraud", "unauthorized", "stolen", "steal", "scam", "theft",
                "identity", "threat", "lawsuit", "legal", "police", "victim"]
df["severe_flag"] = df["Consumer complaint narrative"].str.contains(
    "|".join(SEVERE_WORDS), case=False, na=False).astype(int)

print(f"Average negativity across all complaints: {df['negativity'].mean():.3f}  (0=positive tone, 1=very negative)")
print(f"Share mentioning a high-stakes word: {df['severe_flag'].mean()*100:.1f}%")

**📖 Interpretation:** complaints skew negative on average, which makes sense — people
write in when something went wrong. The high-stakes share tells us what fraction of *all*
complaints mention serious language; this on its own doesn't tell us which *theme* is worst
— that comes next when we aggregate per theme.


## 🏆 Step 6 — Build the final priority ranking (volume × severity)

**Priority formula:** `priority = share_of_complaints(%) × severity(0-1)`.
This is the classic business "priority matrix" idea: a problem that is both **common**
(high volume share) **and** **serious** (high severity) should be fixed first — outranking
a problem that's only common, or only severe, but not both.

We also extract each theme's most distinctive keywords using **TF-IDF** (Term
Frequency - Inverse Document Frequency): a score that rewards words that appear often
*inside* one theme but rarely *across all* themes — this is what lets us label
Theme 7 as being about "unauthorized applications" instead of generic filler words.


In [ ]:
# Cell 5 — Keyword extraction (TF-IDF) + aggregate to theme level
NOISE = {"redacted", "date", "money", "xxxx", "xx", "account", "credit", "card",
         "credit card", "com", "www", "did", "told", "said"}
vec = TfidfVectorizer(max_features=4000, stop_words="english", ngram_range=(1, 2))
tfidf = vec.fit_transform(df["narrative_clean"])
terms = np.array(vec.get_feature_names_out())

def top_keywords(theme_id, n=6):
    rows = np.where(df["theme"].values == theme_id)[0]
    order = np.asarray(tfidf[rows].mean(axis=0)).ravel().argsort()[::-1]
    out = []
    for i in order:
        w = terms[i]
        if w not in NOISE and not any(p in NOISE for p in w.split()):
            out.append(w)
        if len(out) == n:
            break
    return out

g = df.groupby("theme")
theme_tbl = pd.DataFrame({
    "count": g.size(),
    "severity": ((g["negativity"].mean() + g["severe_flag"].mean()) / 2).round(3),
})
theme_tbl["share_%"] = (theme_tbl["count"] / len(df) * 100).round(1)
theme_tbl["priority"] = (theme_tbl["share_%"] * theme_tbl["severity"]).round(2)
theme_tbl["keywords"] = [", ".join(top_keywords(t)) for t in theme_tbl.index]
theme_tbl = theme_tbl.sort_values("priority", ascending=False).reset_index()
theme_tbl.insert(0, "rank", range(1, len(theme_tbl) + 1))

theme_tbl[["rank", "count", "share_%", "severity", "priority", "keywords"]]

**📖 Interpretation of this table:** this is the raw ranking before human-readable AI
names are attached (next step). Notice the top rank is **not** simply the biggest cluster
by count — it's the cluster with the best combination of size *and* severity. This is the
whole point of the priority formula: it prevents a large-but-mild theme from crowding out
a smaller-but-serious one.


## 🏷️ Step 7 — Naming each theme in plain English with an LLM (Gemini)

Cluster numbers and keyword-joins ("Unauthorized / Charges / Dispute") are useful for us,
but not what a business stakeholder wants to read. For each theme, we send its top
keywords plus 3 real example complaints to **Google Gemini** (free tier, model
`gemini-flash-lite-latest`) and ask it to return a short name (≤6 words) and a one-line
description, strictly as JSON so it's easy to parse. If the API call fails for any reason,
the code falls back to the keyword-join name automatically — so the pipeline never breaks
just because a network/API call had a hiccup.

**Why load the saved names here instead of calling the API live in this notebook:** the
naming step already ran once during the real build (`analyze_full.py`) and the results were
saved to `outputs/theme_final_full.csv`. Re-calling a paid/rate-limited API every time this
notebook re-runs would be wasteful and would risk quota limits — so, exactly like the
embeddings, we treat "call the LLM" as an expensive one-time step and reload its saved
output here. The naming *logic* is shown in the code comment below for transparency.


In [ ]:
# Cell 6 — Merge the previously-generated AI names into our live-computed table
# (This proves our live clustering/severity/priority numbers match the saved final
#  results exactly — a validation check that nothing drifted between runs.)
saved_final = pd.read_csv(os.path.join(OUT, "theme_final_full.csv"))

final = theme_tbl.merge(
    saved_final[["theme", "name", "description"]], on="theme", how="left"
).sort_values("rank")

# Validation: does our fresh clustering match the saved run exactly?
match = (final["priority"].round(2).values == saved_final.sort_values("rank")["priority"].round(2).values).all()
print(f"Live-computed priorities match the saved full-run results: {match}")

final[["rank", "name", "share_%", "severity", "priority", "description"]]

**📖 Interpretation:** the validation line confirms our live re-run (same `random_state`,
same formula) reproduces the exact same priority ranking as the original full build — this
is what **reproducibility** means in practice, and it's why we fix random seeds everywhere.
The final table above is the real deliverable: 12 named, ranked, business-readable
problems, each traceable back to real complaint volume and severity numbers.


## 📊 Step 8 — The priority matrix (the headline chart)

A scatter plot where **x = how common** (volume share), **y = how severe**, and
**bubble size = priority**. The top-right quadrant (common AND severe) is where a business
should focus first.


In [ ]:
# Cell 7 — Priority matrix chart, full 112k
plt.figure(figsize=(11, 7))
plt.scatter(final["share_%"], final["severity"], s=final["priority"] * 40,
            c=final["priority"], cmap="Reds", edgecolors="black", alpha=0.85)
for _, r in final.iterrows():
    plt.annotate(str(r["name"]), (r["share_%"], r["severity"]), fontsize=7.5, ha="center")
plt.axvline(final["share_%"].median(), color="grey", ls="--", lw=0.8)
plt.axhline(final["severity"].median(), color="grey", ls="--", lw=0.8)
plt.title("Priority matrix — FULL 112,481 credit-card complaints (top-right = fix first)")
plt.xlabel("Volume (% of complaints)"); plt.ylabel("Severity (0-1)")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "priority_matrix_full_depth.png"), dpi=120)
plt.show()

**📖 Interpretation:** *Unauthorized Charges and Fraud Disputes* sits furthest into the
top-right — it's both the single largest theme (13.4% of all complaints) and among the
most severe (severity 0.78), which is exactly why it ranks #1 overall. *Rewards and
Promotional Offer Disputes* sits bottom-left — common enough to exist as its own theme, but
low severity, so correctly ranked last despite not being the smallest cluster.


## 🔎 Step 9 — Prove it with real examples (never trust a number alone)

Per our validation rule, we always show *real* underlying data behind a summary number,
not just the number itself. Here are genuine complaints from the #1-ranked theme.


In [ ]:
# Cell 8 — Real example complaints from the #1 theme
top_theme_id = int(final.iloc[0]["theme"])
examples = df[df["theme"] == top_theme_id]["Consumer complaint narrative"].head(3)
for i, ex in enumerate(examples, 1):
    print(f"--- Example {i} ---")
    print(str(ex)[:400], "...\n")

**📖 Interpretation:** reading these confirms the cluster genuinely is about
unauthorized charges and fraud disputes — the AI-generated name and description match what
real customers actually wrote, not just what the keyword statistics suggested.


## 💡 Step 10 — Business insights & honest limitations

**What a business should do with this, in priority order:**
1. **Unauthorized Charges and Fraud Disputes (13.4% of volume, most severe)** — invest in
   faster fraud-dispute resolution and clearer proactive fraud alerts; this is the single
   highest-priority fix.
2. **Billing and Merchant Charge Disputes / Payment Processing errors** — a large share of
   volume traces to billing and payment-processing friction — process/tooling fixes here
   affect the most customers per fix.
3. **Identity Theft, Unauthorized Applications, Unexpected Closures** — smaller in volume
   but high severity; each is a compounding trust-and-legal risk worth targeted attention
   even though it won't move the "most complaints" needle much.
4. **Rewards/Promotional Disputes and High Interest/Fees** — lowest priority by this
   ranking; still real, but comparatively lower urgency.

**Limitations, stated honestly:**
- Severity is a **proxy** (sentiment + keyword flags), not verified financial harm or legal
  outcome — a company with real case data could substitute a better severity signal.
- `K=12` was fixed for a fair comparison to the sample run; a production system might tune
  `K` with a coherence metric (e.g. silhouette score) or use HDBSCAN to avoid choosing `K`
  by hand.
- This is single-product (credit card) and English-language complaints only; the same
  pipeline generalizes to other products/languages but hasn't been tested on them here.


## 🎓 Step 11 — Interview-ready recap

**Key terms to know cold:** embedding, sentence-transformer, KMeans, centroid, TF-IDF,
PCA, VADER sentiment, lexicon-based scoring, priority matrix, reproducibility, proxy metric.

**Likely questions + strong short answers:**
- *"How did you find themes without labeled data?"* — "Unsupervised learning: I embedded
  each complaint into a 384-dimension meaning vector with a sentence-transformer, then used
  KMeans to group similar vectors into 12 themes, and labeled each with its top TF-IDF
  keywords plus an LLM-generated name."
- *"How did you decide what to fix first?"* — "Not by volume alone — I combined frequency
  with a severity score (sentiment negativity + high-stakes keyword share) into a priority
  index, visualized as a frequency-vs-severity matrix."
- *"Did results change with more data?"* — "Yes — on an 18k sample, fraud-related
  complaints blurred across 3 overlapping clusters; on the full 112k, they separated into 4
  clean, distinct categories. More data per cluster gave more stable centroids."
- *"What's a limitation of your approach?"* — "Severity is a proxy from sentiment and
  keyword matching, not verified financial harm — I state that explicitly rather than
  present it as ground truth."

---
*This notebook is the full-scale, deep-depth companion to `phase3_engine.ipynb` (the 18k
prototype). Together with `phase1_data.ipynb` and `phase2_clean.ipynb`, they document the
complete, real, defensible build of the Insight Engine.*
